# Word Boundary Benchmark

This notebook documents the sentence word-audio helper benchmark for Read-Along AI and separates it from the recorded child-speech dataset used for ASR and MiniCPM judging. The app asks VoxCPM to read a whole sentence once, then slices that sentence audio into clickable word clips. The key product requirement is simple: when a child taps `the` in `The dog ran fast.`, the helper should play the `the` from that sentence, not a stale clip from another sentence and not a neighboring partial word.

The real labeled data tells us how the reading evaluator performs on recorded child word utterances. The synthetic comma-pause benchmark tells us whether sentence-level TTS can be sliced into clean word clips. They are related parts of the product, but they are not the same test.

## Why Comma-Separated TTS

For word help, generating each word independently is slow and can make the local UI feel stuck. The current approach generates one sentence-shaped TTS sample and asks the TTS prompt to include comma-like pauses between words. Those pauses become measurable silence gaps in the WAV file.

That gives us a lightweight alignment signal:

1. Normalize the sentence into clickable word keys.
2. Generate or simulate audio with short silent gaps between words.
3. Detect the largest internal silence gaps.
4. Use those gaps as word boundaries.

The old fallback did not listen to the audio. It split the total duration by word length, which is attractive because it is simple, but it breaks when text length and spoken duration do not match.

In [ ]:
from __future__ import annotations

import io
import json
import math
import sys
import wave
from array import array
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import word_boundaries

SAMPLE_RATE = 16_000

## Provided Labeled Dataset

The project includes a real labeled child-speech dataset. That dataset is the right evidence for ASR behavior and the MiniCPM phonetic judge. It is not, by itself, a word-boundary benchmark because the cleaned files are isolated word recordings rather than sentence-level TTS clips with known internal word boundaries.

In this repo, the labeled assets are:

- `data/processed_audio/jane_cleaned/`: cleaned per-word recordings.
- `data/baseline_results.csv`: target word, ASR transcript, and strict-match result.
- `data/train.jsonl`: target/transcript examples labeled `True` or `False` for the MiniCPM phonetic judge.

In [ ]:
labeled_audio_dir = REPO_ROOT / "data" / "processed_audio" / "jane_cleaned"
baseline_results_path = REPO_ROOT / "data" / "baseline_results.csv"
train_jsonl_path = REPO_ROOT / "data" / "train.jsonl"

labeled_audio_files = sorted(labeled_audio_dir.glob("*.wav"))
baseline = pd.read_csv(baseline_results_path)
train_examples = [json.loads(line) for line in train_jsonl_path.read_text().splitlines() if line.strip()]
train = pd.DataFrame(train_examples)

provided_summary = pd.DataFrame(
    [
        {
            "Artifact": "Cleaned child word recordings",
            "Path": str(labeled_audio_dir.relative_to(REPO_ROOT)),
            "Rows / files": len(labeled_audio_files),
            "Used for": "Recorded speech examples",
        },
        {
            "Artifact": "Baseline ASR results",
            "Path": str(baseline_results_path.relative_to(REPO_ROOT)),
            "Rows / files": len(baseline),
            "Used for": "Strict ASR match evaluation",
        },
        {
            "Artifact": "MiniCPM judge labels",
            "Path": str(train_jsonl_path.relative_to(REPO_ROOT)),
            "Rows / files": len(train),
            "Used for": "Phonetic match supervision",
        },
    ]
)

provided_summary

In [ ]:
label_counts = train["output"].value_counts().rename_axis("MiniCPM label").reset_index(name="Count")
baseline_summary = pd.DataFrame(
    [
        {
            "Metric": "Strict ASR matches",
            "Value": int(baseline["Strict Match"].sum()),
            "Total": len(baseline),
            "Rate": baseline["Strict Match"].mean(),
        },
        {
            "Metric": "Strict ASR misses",
            "Value": int((~baseline["Strict Match"]).sum()),
            "Total": len(baseline),
            "Rate": (~baseline["Strict Match"]).mean(),
        },
    ]
)

display(baseline_summary.style.format({"Rate": "{:.1%}"}).hide(axis="index"))
display(label_counts)

## Boundary Benchmark Dataset

The boundary benchmark below is synthetic. That is intentional, but it should be named clearly. We are isolating one behavior that the recorded word dataset does not label: whether internal sentence-audio cuts land inside known silence gaps.

Each row defines a sentence, spoken word durations, and manually labeled silence gaps. The three cases cover:

- Even text length and even audio timing.
- Very uneven word lengths with equal audio timing.
- Similar word lengths with uneven audio timing.

In [ ]:
@dataclass(frozen=True)
class SyntheticBoundaryCase:
    name: str
    sentence: str
    word_durations: tuple[float, ...]
    gap_durations: tuple[float, ...]

    @property
    def manual_gaps(self) -> list[tuple[float, float]]:
        elapsed = 0.0
        gaps: list[tuple[float, float]] = []
        for index, duration in enumerate(self.word_durations):
            elapsed += duration
            if index < len(self.gap_durations):
                gap_start = elapsed
                elapsed += self.gap_durations[index]
                gaps.append((gap_start, elapsed))
        return gaps

    @property
    def total_duration(self) -> float:
        return sum(self.word_durations) + sum(self.gap_durations)


CASES = (
    SyntheticBoundaryCase(
        name="equal_text_equal_audio",
        sentence="cat, dog, red, hat",
        word_durations=(0.22, 0.24, 0.24, 0.24),
        gap_durations=(0.12, 0.14, 0.14),
    ),
    SyntheticBoundaryCase(
        name="uneven_text_equal_audio",
        sentence="a, elephant, ox, rhinoceros",
        word_durations=(0.22, 0.24, 0.24, 0.24),
        gap_durations=(0.12, 0.14, 0.14),
    ),
    SyntheticBoundaryCase(
        name="uneven_audio_even_text",
        sentence="blue, green, white, black",
        word_durations=(0.14, 0.33, 0.18, 0.29),
        gap_durations=(0.10, 0.16, 0.12),
    ),
)

dataset_rows = []
for case in CASES:
    dataset_rows.append(
        {
            "Case": case.name,
            "Sentence": case.sentence,
            "Words": word_boundaries.sentence_tts_words(case.sentence),
            "Word durations (s)": case.word_durations,
            "Manual gaps (s)": case.manual_gaps,
            "Total duration (s)": round(case.total_duration, 3),
        }
    )

pd.DataFrame(dataset_rows)

## Audio Fixture Generation

The benchmark uses sine tones for spoken words and zeros for comma pauses. This lets us know exactly where the intended boundaries are without requiring a live TTS model call.

In [ ]:
def tone(duration: float, amplitude: int = 3000, frequency: float = 220.0) -> array:
    samples = array("h")
    for index in range(int(SAMPLE_RATE * duration)):
        samples.append(int(amplitude * math.sin(2 * math.pi * frequency * index / SAMPLE_RATE)))
    return samples


def silence(duration: float) -> array:
    return array("h", [0] * int(SAMPLE_RATE * duration))


def wav_bytes(segments: list[array]) -> bytes:
    pcm = array("h")
    for segment in segments:
        pcm.extend(segment)

    buffer = io.BytesIO()
    with wave.open(buffer, "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(SAMPLE_RATE)
        wav_file.writeframes(pcm.tobytes())
    return buffer.getvalue()


def case_audio(case: SyntheticBoundaryCase) -> bytes:
    segments: list[array] = []
    for index, duration in enumerate(case.word_durations):
        segments.append(tone(duration, frequency=220.0 + 40.0 * index))
        if index < len(case.gap_durations):
            segments.append(silence(case.gap_durations[index]))
    return wav_bytes(segments)


fixture_rows = [
    {"Case": case.name, "Audio bytes": len(case_audio(case)), "Manual boundaries": len(case.manual_gaps)}
    for case in CASES
]
pd.DataFrame(fixture_rows)

## Competing Boundary Methods

The current method calls `word_boundaries.word_boundary_timestamps`, which prefers detected silence gaps. The previous method is reproduced here as a proportional splitter: each word receives a share of total audio duration based on character count.

In [ ]:
def current_silence_gap_boundaries(case: SyntheticBoundaryCase) -> list[float]:
    stamps = word_boundaries.word_boundary_timestamps(case.sentence, case_audio(case))
    return [end for _word, _start, end in stamps[:-1]]


def previous_proportional_boundaries(case: SyntheticBoundaryCase) -> list[float]:
    words = word_boundaries.sentence_tts_words(case.sentence)
    weights = [max(len(word), 1) for word in words]
    total_weight = sum(weights)
    elapsed_weight = 0
    boundaries: list[float] = []
    for weight in weights[:-1]:
        elapsed_weight += weight
        boundaries.append(case.total_duration * elapsed_weight / total_weight)
    return boundaries


comparison_rows = []
for case in CASES:
    for boundary_index, (gap_start, gap_end) in enumerate(case.manual_gaps, start=1):
        midpoint = (gap_start + gap_end) / 2
        current = current_silence_gap_boundaries(case)[boundary_index - 1]
        previous = previous_proportional_boundaries(case)[boundary_index - 1]
        comparison_rows.extend(
            [
                {
                    "Case": case.name,
                    "Boundary": boundary_index,
                    "Method": "Current silence-gap splitter",
                    "Boundary time (s)": current,
                    "Manual gap start (s)": gap_start,
                    "Manual gap end (s)": gap_end,
                    "Hit manual gap": gap_start <= current <= gap_end,
                    "Abs error to midpoint (s)": abs(current - midpoint),
                },
                {
                    "Case": case.name,
                    "Boundary": boundary_index,
                    "Method": "Previous proportional splitter",
                    "Boundary time (s)": previous,
                    "Manual gap start (s)": gap_start,
                    "Manual gap end (s)": gap_end,
                    "Hit manual gap": gap_start <= previous <= gap_end,
                    "Abs error to midpoint (s)": abs(previous - midpoint),
                },
            ]
        )

comparison = pd.DataFrame(comparison_rows)
comparison

## Aggregate Results

In [ ]:
summary = (
    comparison.groupby("Method", as_index=False)
    .agg(
        Hits=("Hit manual gap", "sum"),
        Total=("Hit manual gap", "count"),
        Mean_abs_error_s=("Abs error to midpoint (s)", "mean"),
        Max_abs_error_s=("Abs error to midpoint (s)", "max"),
    )
    .assign(Hit_rate=lambda df: df["Hits"] / df["Total"])
    [["Method", "Hits", "Total", "Hit_rate", "Mean_abs_error_s", "Max_abs_error_s"]]
)

summary.style.format(
    {
        "Hit_rate": "{:.1%}",
        "Mean_abs_error_s": "{:.4f}",
        "Max_abs_error_s": "{:.4f}",
    }
).hide(axis="index")

Expected local result from the current repository:

| Method | Manual-gap hits | Hit rate | Mean boundary error | Max boundary error |
|---|---:|---:|---:|---:|
| Current silence-gap splitter | 9/9 | 100.0% | 0.0000s | 0.0000s |
| Previous proportional splitter | 5/9 | 55.6% | 0.0928s | 0.3281s |

## Interpretation

The current method wins because it uses an acoustic feature we intentionally create: comma-like silence between words. On this benchmark, every detected internal boundary lands inside the manual silence gap.

The proportional splitter fails in the two cases we care about most. In `a, elephant, ox, rhinoceros`, text length varies dramatically while spoken timing is almost even. In `blue, green, white, black`, text length is fairly even but spoken timing varies. Both are normal TTS behaviors, which makes character-weight splitting a weak proxy for alignment.

The product consequence is practical: sentence-level prewarming can stay fast, but word clicks are less likely to play a neighboring fragment such as `the ca` when the user taps `the`.

## Current App Guardrail

The benchmark above covers boundary quality. The app also now scopes cached word clips by `(sentence, word)` rather than only by `word`. That prevents a shared word like `the` from being reused across `The cat sat.` and `The dog ran fast.`.

Together, these two changes address the two observed failure modes:

- Better slicing: avoid partial neighboring audio within a sentence.
- Sentence-scoped cache: avoid stale shared-word audio across sentences.

## Caveat

This is still a synthetic benchmark. It is a strong unit-style check for comma-pause boundary behavior, not a full forced-alignment evaluation over real VoxCPM sentence outputs. The next useful benchmark would save real generated sentence audio, manually label the word gaps, and run the same metrics against those clips.